# 03 OCR - 00: Pretrained Gate (YOLO11m Only)

Yolo-only base notebook for `ultralytics_yolo11m_baseline` on `wm_polygon` and `wm_bbox` crops.
This version keeps only core Stage A and preview blocks (no Stage A+ sweep / failure dumps / delta gate).
Artifacts are written with `_yolo11m_model_only` suffix.


## 0. Импорты и локальные пути (без Colab)

Верхняя ячейка содержит только импорты и локальную конфигурацию окружения.

In [1]:
import sys, json, time, shutil
from datetime import datetime
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import segmentation_models_pytorch as smp

ROOT = Path("../..").resolve()
DATA_ROOT = ROOT / "WaterMetricsDATA"
RESULTS_DIR = ROOT / "results"

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import importlib
import models.data.ocr_dataset as _ocr_dataset
importlib.reload(_ocr_dataset)
prepare_ocr_crops = _ocr_dataset.prepare_ocr_crops
load_ocr_crops = _ocr_dataset.load_ocr_crops
from models.metrics.ocr_metrics import evaluate_ocr_batch, build_ocr_comparison_row, append_ocr_comparison_row
from models.utils.orientation import dual_read_inference

c:\Users\alike\WaterMeterCV\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Конфигурация данных и ROI

Базовые пути, веса и параметры для генерации OCR crops.

In [2]:
CROPS_ROOT = DATA_ROOT / "ocr_crops"
WM_PATH = DATA_ROOT / "waterMeterDataset/WaterMeters"
WM_ROI_YOLO_WEIGHTS = ROOT / "models/weights/roi_yolo/wm_yolo_roi_20260412_230832/weights/best.pt"
WM_POLYGON_UNET_WEIGHTS = ROOT / "models/weights/roi_unet/wm_unet_20260415_004934_best.pt"
WM_POLYGON_SOURCE = "unet_best"  # "unet_best" or "gt"
WM_POLYGON_INPUT_SIZE = 512
WM_POLYGON_MASK_THRESHOLD = 0.5
WM_POLYGON_MIN_AREA = 128.0
WM_POLYGON_FALLBACK_TO_GT_ON_MISS = False
WM_BBOX_SOURCE = "yolo"  # "yolo" or "gt"
WM_BBOX_FALLBACK_TO_GT_ON_MISS = False
WM_LABEL_MODE = "wm_fraction_aware"  # "source" or "wm_fraction_aware"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TARGET_CANDIDATE = "ultralytics_yolo11m_baseline"
TARGET_SUFFIX = "_yolo11m_model_only"

## 2. Детекторы ROI и подготовка crop-датасета

Здесь определяются билдеры ROI-детекторов и выполняется пересоздание `ocr_crops` с метаданными.

In [3]:
def build_wm_roi_detector(weights_path: Path, source: str = WM_BBOX_SOURCE):
    if source != "yolo":
        return None
    if not weights_path.exists():
        print(f"[prepare_ocr_crops] YOLO ROI weights not found: {weights_path}. Fallback to GT bbox.")
        return None

    try:
        from ultralytics import YOLO
        model = YOLO(str(weights_path))
    except Exception as exc:
        print(f"[prepare_ocr_crops] Failed to load YOLO ROI model: {exc}. Fallback to GT bbox.")
        return None

    def _detect(img_bgr):
        try:
            result = model.predict(img_bgr, verbose=False, conf=0.001, imgsz=640, max_det=16)[0]
        except Exception:
            return None

        boxes = getattr(result, "boxes", None)
        if boxes is None or len(boxes) == 0:
            return None

        try:
            idx = int(boxes.conf.argmax().item()) if hasattr(boxes, "conf") else 0
        except Exception:
            idx = 0

        b = boxes.xywhn[idx]
        return float(b[0]), float(b[1]), float(b[2]), float(b[3])

    return _detect

In [4]:
def build_wm_unet_polygon_detector(
    source: str = WM_POLYGON_SOURCE,
    weights_path: Path = WM_POLYGON_UNET_WEIGHTS,
    img_size: int = WM_POLYGON_INPUT_SIZE,
    mask_threshold: float = WM_POLYGON_MASK_THRESHOLD,
    min_area: float = WM_POLYGON_MIN_AREA,
):
    if source != "unet_best":
        return None, None, "gt"

    if not weights_path.exists():
        raise FileNotFoundError(f"WM U-Net checkpoint not found: {weights_path}")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = smp.Unet(
        encoder_name="resnet34",
        encoder_weights=None,
        in_channels=3,
        classes=1,
    ).to(device)

    try:
        state_dict = torch.load(weights_path, map_location=device, weights_only=True)
    except TypeError:
        state_dict = torch.load(weights_path, map_location=device)

    model.load_state_dict(state_dict)
    model.eval()

    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(1, 1, 3)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(1, 1, 3)

    def _detect(img_bgr):
        if img_bgr is None or img_bgr.size == 0:
            return None

        h, w = img_bgr.shape[:2]
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        resized = cv2.resize(img_rgb, (img_size, img_size), interpolation=cv2.INTER_LINEAR)

        x = resized.astype(np.float32) / 255.0
        x = (x - mean) / std
        x = torch.from_numpy(x).permute(2, 0, 1).unsqueeze(0).to(device)

        with torch.no_grad():
            pred = model(x)
            mask = (torch.sigmoid(pred[0, 0]).cpu().numpy() >= float(mask_threshold)).astype(np.uint8)

        if mask.max() == 0:
            return None

        mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
        mask = (mask > 0).astype(np.uint8) * 255

        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            return None

        cnt = max(contours, key=cv2.contourArea)
        if cv2.contourArea(cnt) < float(min_area):
            return None

        box = cv2.boxPoints(cv2.minAreaRect(cnt))
        polygon = []
        denom_w = float(max(w - 1, 1))
        denom_h = float(max(h - 1, 1))
        for px, py in box:
            x_norm = float(np.clip(float(px) / denom_w, 0.0, 1.0))
            y_norm = float(np.clip(float(py) / denom_h, 0.0, 1.0))
            polygon.append((x_norm, y_norm))
        return polygon

    source_desc = f"unet_best ({weights_path.name})"
    return _detect, weights_path, source_desc

In [5]:
wm_polygon_detector, wm_polygon_checkpoint, wm_polygon_source_used = build_wm_unet_polygon_detector(
    source=WM_POLYGON_SOURCE,
    weights_path=WM_POLYGON_UNET_WEIGHTS,
)

if CROPS_ROOT.exists():
    shutil.rmtree(CROPS_ROOT)
    print(f"[prepare_ocr_crops] Deleted existing dataset root: {CROPS_ROOT}")
else:
    print(f"[prepare_ocr_crops] Nothing to delete: {CROPS_ROOT}")

wm_roi_detector = build_wm_roi_detector(WM_ROI_YOLO_WEIGHTS, source=WM_BBOX_SOURCE)
if wm_roi_detector is None:
    raise RuntimeError(f"wm_yolo_roi detector is required for OCR crops generation. Weights: {WM_ROI_YOLO_WEIGHTS}")

create_t0 = time.perf_counter()
prepare_ocr_crops(
    WM_PATH,
    CROPS_ROOT,
    roi_polygon_detector=wm_polygon_detector,
    fallback_to_gt_polygon_on_miss=WM_POLYGON_FALLBACK_TO_GT_ON_MISS,
    roi_detector=wm_roi_detector,
    fallback_to_gt_on_miss=WM_BBOX_FALLBACK_TO_GT_ON_MISS,
    label_mode=WM_LABEL_MODE,
)
create_seconds = time.perf_counter() - create_t0
print("wm_polygon source:", wm_polygon_source_used)
if wm_polygon_checkpoint is not None:
    print("wm_polygon checkpoint:", wm_polygon_checkpoint)
bbox_source_used = ("yolo+gt_fallback" if WM_BBOX_FALLBACK_TO_GT_ON_MISS else "yolo") if wm_roi_detector is not None else "gt"
print("wm_bbox source:", bbox_source_used)
print("wm_polygon fallback_to_gt_on_miss:", WM_POLYGON_FALLBACK_TO_GT_ON_MISS)
print("wm_bbox fallback_to_gt_on_miss:", WM_BBOX_FALLBACK_TO_GT_ON_MISS)
print("wm label mode:", WM_LABEL_MODE)
wm_poly_test = load_ocr_crops(CROPS_ROOT / "wm_polygon", "test")
wm_bbox_test = load_ocr_crops(CROPS_ROOT / "wm_bbox", "test")
generated_at = datetime.now().isoformat(timespec="seconds")
readme_path = CROPS_ROOT / "README.md"
readme_lines = [
    "# OCR Crops Dataset" ,
    "" ,
    f"- generated_at: {generated_at}" ,
    f"- generation_seconds: {create_seconds:.3f}" ,
    f"- source_watermeter_path: {WM_PATH}" ,
    f"- roi_detector_model: {WM_ROI_YOLO_WEIGHTS.parent.parent.name}" ,
    f"- roi_detector_weights: {WM_ROI_YOLO_WEIGHTS}" ,
    f"- polygon_source: {wm_polygon_source_used}" ,
    f"- polygon_detector_checkpoint: {wm_polygon_checkpoint if wm_polygon_checkpoint is not None else 'none'}" ,
    f"- bbox_source: {WM_BBOX_SOURCE}" ,
    f"- bbox_source_mode: {bbox_source_used}" ,
    f"- fallback_to_gt_polygon_on_miss: {WM_POLYGON_FALLBACK_TO_GT_ON_MISS}" ,
    f"- fallback_to_gt_on_miss: {WM_BBOX_FALLBACK_TO_GT_ON_MISS}" ,
    f"- label_mode: {WM_LABEL_MODE}" ,
    f"- wm_polygon_test_samples: {len(wm_poly_test)}" ,
    f"- wm_bbox_test_samples: {len(wm_bbox_test)}" ,
]
readme_path.write_text("\n".join(readme_lines) + "\n", encoding="utf-8")
print(f"[prepare_ocr_crops] Dataset recreated in {create_seconds:.2f}s")
print(f"[prepare_ocr_crops] Wrote dataset README: {readme_path}")

[prepare_ocr_crops] Deleted existing dataset root: C:\Users\alike\WaterMeterCV\WaterMetricsDATA\ocr_crops
wm_polygon source: unet_best (wm_unet_20260415_004934_best.pt)
wm_polygon checkpoint: C:\Users\alike\WaterMeterCV\models\weights\roi_unet\wm_unet_20260415_004934_best.pt
wm_bbox source: yolo
wm_polygon fallback_to_gt_on_miss: False
wm_bbox fallback_to_gt_on_miss: False
wm label mode: wm_fraction_aware
[prepare_ocr_crops] Dataset recreated in 207.70s
[prepare_ocr_crops] Wrote dataset README: C:\Users\alike\WaterMeterCV\WaterMetricsDATA\ocr_crops\README.md


## 3. OCR эвристики и ориентация

В этом блоке находятся эвристики для фильтрации боксов, выбор ориентации 0°/180° и сборка финального предиктора.

In [19]:
# Toggle preprocessing strategy for OCR engines.
# Options: "none", "gray_clahe"
PREPROCESS_MODE = "none"
MAX_READING_DIGITS = 10
LEADING_ZERO_ORIENTATION_MIN = 3

# Ultralytics heuristics
ULTRA_LAST_DRUM_X_ALIGN_FACTOR = 0.55
ULTRA_LAST_DRUM_MIN_Y_GAP_FACTOR = 0.25
ULTRA_OVERLAP_IOA_MIN = 0.65
ULTRA_OVERLAP_CENTER_FACTOR = 0.55

# Red-cluster orientation heuristic (computed only inside Ultralytics bboxes)
RED_CLUSTER_MIN_COVERAGE = 0.0025
RED_CLUSTER_LEFT_MAX_X = 0.43
RED_CLUSTER_RIGHT_MIN_X = 0.57
RED_BBOX_MIN_ACTIVE_PIXELS = 64

# Statistical tail heuristic
LONG_TAIL_ZERO_MIN_DIGITS = 8
LONG_TAIL_ZERO_MIN_SUFFIX = 2

# Orientation voting weights (sum ~= 1.0)
RED_BBOX_VOTE_WEIGHT = 0.85
STAT_TAIL_VOTE_WEIGHT = 0.09
LEADING_ZERO_VOTE_WEIGHT = 0.06

# Make red-box prior slightly stricter (~8%).
RED_BBOX_STRICTNESS = 0.08

ULTRALYTICS_MODEL_CACHE = {}

In [18]:
def preprocess_for_ocr(image_bgr, mode=PREPROCESS_MODE):
    if mode == "none":
        return image_bgr

    if mode == "gray_clahe":
        gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        gray = clahe.apply(gray)
        return cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)

    raise ValueError(f"Unsupported preprocess mode: {mode!r}")


def wrap_predictor_with_preprocess(predictor, mode=PREPROCESS_MODE):
    def _predict(image_bgr):
        preprocessed = preprocess_for_ocr(image_bgr, mode=mode)
        return predictor(preprocessed)

    return _predict


def digits_only(text):
    return "".join(ch for ch in str(text) if ch.isdigit())


def safe_mean(values):
    return float(np.mean(values)) if values else 0.0


def apply_max_digits_heuristic(pred, conf, max_digits=MAX_READING_DIGITS):
    pred_digits = digits_only(pred)
    try:
        conf_val = float(conf)
    except Exception:
        conf_val = 0.0

    if len(pred_digits) <= max_digits:
        return pred_digits, conf_val

    # Penalize overlong outputs and keep only the leftmost max_digits.
    overflow = len(pred_digits) - max_digits
    penalty = max(0.0, 1.0 - (overflow / max_digits))
    return pred_digits[:max_digits], conf_val * penalty


def wrap_predictor_with_max_digits(predictor, max_digits=MAX_READING_DIGITS):
    def _predict(image_bgr):
        pred, conf = predictor(image_bgr)
        return apply_max_digits_heuristic(pred, conf, max_digits=max_digits)

    return _predict


def leading_zero_count(text):
    d = digits_only(text)
    n = 0
    for ch in d:
        if ch == "0":
            n += 1
        else:
            break
    return n


def normalize_digits_for_stats(text):
    d = digits_only(text).lstrip("0")
    return d if d else "0"


def trailing_zero_count(text):
    n = 0
    for ch in reversed(str(text)):
        if ch == "0":
            n += 1
        else:
            break
    return n


def is_long_tail_zero_pattern(text):
    norm = normalize_digits_for_stats(text)
    return len(norm) >= LONG_TAIL_ZERO_MIN_DIGITS and trailing_zero_count(norm) >= LONG_TAIL_ZERO_MIN_SUFFIX


def ultralytics_digit_rank(digit):
    d = int(digit)
    # Drum rollover rule: 0 is treated as greater than 9.
    return 10 if d == 0 else d


def bbox_intersection_area(a, b):
    x_left = max(a["x1"], b["x1"])
    y_top = max(a["y1"], b["y1"])
    x_right = min(a["x2"], b["x2"])
    y_bottom = min(a["y2"], b["y2"])
    if x_right <= x_left or y_bottom <= y_top:
        return 0.0
    return float((x_right - x_left) * (y_bottom - y_top))


def bbox_area(d):
    return float(max(d["x2"] - d["x1"], 0.0) * max(d["y2"] - d["y1"], 0.0))


def boxes_are_nested_or_almost_nested(a, b):
    inter = bbox_intersection_area(a, b)
    if inter <= 0.0:
        return False

    area_a = max(bbox_area(a), 1e-6)
    area_b = max(bbox_area(b), 1e-6)
    ioa_small = inter / min(area_a, area_b)

    if ioa_small < ULTRA_OVERLAP_IOA_MIN:
        return False

    x_close = abs(a["cx"] - b["cx"]) <= ULTRA_OVERLAP_CENTER_FACTOR * max(a["w"], b["w"], 1e-6)
    y_close = abs(a["cy"] - b["cy"]) <= ULTRA_OVERLAP_CENTER_FACTOR * max(a["h"], b["h"], 1e-6)
    return bool(x_close and y_close)

In [8]:
# No-red orientation prior for short integer-only readings.
# Rule: if reading has < 8 digits, starts with non-zero and ends with zero,
# then this orientation is suspicious and likely upside-down.
NO_RED_SHORT_READING_MAX_DIGITS = 7
NO_RED_SHORT_READING_VOTE_WEIGHT = 0.07


def is_no_red_upside_down_pattern(text, max_digits=NO_RED_SHORT_READING_MAX_DIGITS):
    d = digits_only(text)
    if not d:
        return False
    if len(d) > int(max_digits):
        return False
    return d[0] != "0" and d[-1] == "0"

In [9]:
def apply_ultralytics_overlap_heuristic(detections):
    dets = sorted(detections, key=lambda d: d["cx"])
    n = len(dets)
    if n < 2:
        return dets, {"applied": False, "reason": "not_enough_boxes"}

    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a_idx, b_idx):
        ra, rb = find(a_idx), find(b_idx)
        if ra != rb:
            parent[rb] = ra

    for i in range(n):
        for j in range(i + 1, n):
            if boxes_are_nested_or_almost_nested(dets[i], dets[j]):
                union(i, j)

    groups = {}
    for idx in range(n):
        root = find(idx)
        groups.setdefault(root, []).append(dets[idx])

    filtered = []
    applied_count = 0
    for _, group in groups.items():
        if len(group) == 1:
            filtered.append(group[0])
            continue

        applied_count += 1
        zeros = [g for g in group if int(g["digit"]) == 0]
        if zeros:
            chosen = max(zeros, key=lambda g: (float(g["conf"]), -bbox_area(g)))
        else:
            chosen = max(group, key=lambda g: (ultralytics_digit_rank(g["digit"]), float(g["conf"])))
        filtered.append(chosen)

    filtered.sort(key=lambda d: d["cx"])
    return filtered, {
        "applied": applied_count > 0,
        "reason": "overlap_group",
        "group_count": int(applied_count),
    }


def apply_ultralytics_last_drum_heuristic(detections):
    dets = sorted(detections, key=lambda d: d["cx"])
    if len(dets) < 2:
        return dets, {"applied": False, "reason": "not_enough_boxes"}

    a, b = dets[-2], dets[-1]
    w_ref = max(a["w"], b["w"], 1e-6)
    h_ref = max(a["h"], b["h"], 1e-6)

    x_close = abs(a["cx"] - b["cx"]) <= ULTRA_LAST_DRUM_X_ALIGN_FACTOR * w_ref
    y_split = abs(a["cy"] - b["cy"]) >= ULTRA_LAST_DRUM_MIN_Y_GAP_FACTOR * h_ref
    x_overlap = min(a["x2"], b["x2"]) > max(a["x1"], b["x1"])

    if not (x_close and y_split and x_overlap):
        return dets, {"applied": False, "reason": "not_vertical_pair"}

    # Prefer stronger digit first (with 0 > 9), then confidence.
    rank_a = ultralytics_digit_rank(a["digit"])
    rank_b = ultralytics_digit_rank(b["digit"])
    if rank_a != rank_b:
        chosen = a if rank_a > rank_b else b
        reason = "digit_rank"
    elif a["conf"] != b["conf"]:
        chosen = a if a["conf"] >= b["conf"] else b
        reason = "confidence"
    elif a["cy"] != b["cy"]:
        chosen = a if a["cy"] < b["cy"] else b
        reason = "upper_tiebreak"
    else:
        chosen = a
        reason = "left_tiebreak"

    dropped = b if chosen is a else a
    filtered = dets[:-2] + [chosen]
    return filtered, {
        "applied": True,
        "reason": reason,
        "kept_digit": int(chosen["digit"]),
        "dropped_digit": int(dropped["digit"]),
    }

In [10]:
def get_ultralytics_baseline_model():
    cache_key = "ultralytics_yolo11m_baseline"
    if cache_key in ULTRALYTICS_MODEL_CACHE:
        return ULTRALYTICS_MODEL_CACHE[cache_key]

    try:
        from ultralytics import YOLO

        yolo_run_dir = ROOT / "models/weights/baseline_yolo/yolo11m_20260414_194809"
        yolo_weight_candidates = [
            yolo_run_dir / "weights/best.pt",
            yolo_run_dir / "weights/last.pt",
            yolo_run_dir if yolo_run_dir.suffix.lower() == ".pt" else None,
        ]
        yolo_weight_path = next((p for p in yolo_weight_candidates if p is not None and p.exists()), None)
        if yolo_weight_path is None:
            ULTRALYTICS_MODEL_CACHE[cache_key] = None
            return None

        model = YOLO(str(yolo_weight_path))
        ULTRALYTICS_MODEL_CACHE[cache_key] = model
        return model
    except Exception:
        ULTRALYTICS_MODEL_CACHE[cache_key] = None
        return None


def extract_ultralytics_digit_detections(image_bgr, model=None):
    if image_bgr is None:
        return []

    if model is None:
        model = get_ultralytics_baseline_model()
    if model is None:
        return []

    try:
        result = model.predict(image_bgr, verbose=False, imgsz=320, max_det=32)[0]
    except Exception:
        return []

    boxes = getattr(result, "boxes", None)
    if boxes is None or len(boxes) == 0:
        return []

    digit_mask = boxes.cls <= 9
    digit_boxes = boxes[digit_mask]
    if len(digit_boxes) == 0:
        return []

    sorted_idx = digit_boxes.xywh[:, 0].argsort()
    detections = []
    for i in sorted_idx:
        xyxy = digit_boxes.xyxy[i].tolist()
        xywh = digit_boxes.xywh[i].tolist()
        if len(xyxy) < 4 or len(xywh) < 4:
            continue

        x1, y1, x2, y2 = [float(v) for v in xyxy[:4]]
        cx, cy, w, h = [float(v) for v in xywh[:4]]
        digit = int(digit_boxes.cls[i].item())
        score = float(digit_boxes.conf[i].item()) if hasattr(digit_boxes, "conf") else 0.0

        detections.append(
            {
                "digit": digit,
                "conf": score,
                "score": score,
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
                "cx": cx,
                "cy": cy,
                "w": w,
                "h": h,
            }
        )

    detections.sort(key=lambda d: d["cx"])
    return detections

In [11]:
def estimate_red_horizontal_cluster_in_bboxes(
    image_bgr,
    detections,
    min_coverage=RED_CLUSTER_MIN_COVERAGE,
    min_active_pixels=RED_BBOX_MIN_ACTIVE_PIXELS,
):
    if image_bgr is None or image_bgr.size == 0 or not detections:
        return None

    b, g, r = cv2.split(image_bgr.astype(np.float32))
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    s = hsv[:, :, 1].astype(np.float32)
    v = hsv[:, :, 2].astype(np.float32)
    red_score = (r - np.maximum(g, b)) + 0.25 * s

    h, w = image_bgr.shape[:2]
    weighted_x_sum = 0.0
    weight_total = 0.0
    active_pixels = 0

    for d in detections:
        x1 = max(0, min(w - 1, int(round(d["x1"]))))
        y1 = max(0, min(h - 1, int(round(d["y1"]))))
        x2 = max(0, min(w - 1, int(round(d["x2"]))))
        y2 = max(0, min(h - 1, int(round(d["y2"]))))
        if x2 <= x1 or y2 <= y1:
            continue

        rr = r[y1 : y2 + 1, x1 : x2 + 1]
        gg = g[y1 : y2 + 1, x1 : x2 + 1]
        bb = b[y1 : y2 + 1, x1 : x2 + 1]
        vv = v[y1 : y2 + 1, x1 : x2 + 1]
        score_patch = red_score[y1 : y2 + 1, x1 : x2 + 1]

        base_mask = (vv >= 30.0) & (rr >= 40.0) & (rr >= gg * 1.02) & (rr >= bb * 1.02)
        if not np.any(base_mask):
            continue

        vals = score_patch[base_mask]
        thr = max(float(np.percentile(vals, 85.0)), 10.0)
        strong = base_mask & (score_patch >= thr)
        if not np.any(strong):
            continue

        _, xs = np.where(strong)
        weights = np.maximum(score_patch[strong] - thr, 0.0) + 1e-3
        weighted_x_sum += float(np.sum((xs.astype(np.float32) + float(x1)) * weights))
        weight_total += float(np.sum(weights))
        active_pixels += int(np.sum(strong))

    if active_pixels < int(min_active_pixels) or weight_total <= 1e-6:
        return None

    denom = float(max(w - 1, 1))
    x_norm = float(weighted_x_sum / (weight_total * denom))
    coverage = float(active_pixels / float(max(h * w, 1)))
    if coverage < float(min_coverage):
        return None

    return {"x_norm": x_norm, "coverage": coverage, "active_pixels": int(active_pixels)}


def get_stricter_red_bbox_thresholds(strictness=RED_BBOX_STRICTNESS):
    center = 0.5
    s = float(np.clip(strictness, 0.0, 0.30))
    left_dist = max(center - RED_CLUSTER_LEFT_MAX_X, 0.0)
    right_dist = max(RED_CLUSTER_RIGHT_MIN_X - center, 0.0)

    left_thr = RED_CLUSTER_LEFT_MAX_X - left_dist * s
    right_thr = RED_CLUSTER_RIGHT_MIN_X + right_dist * s
    min_cov = RED_CLUSTER_MIN_COVERAGE * (1.0 + s)
    min_pixels = int(np.ceil(RED_BBOX_MIN_ACTIVE_PIXELS * (1.0 + s)))
    return left_thr, right_thr, min_cov, min_pixels


def red_bbox_orientation_prior(image_bgr):
    detections = extract_ultralytics_digit_detections(image_bgr)
    if not detections:
        return None

    detections, _ = apply_ultralytics_overlap_heuristic(detections)
    detections, _ = apply_ultralytics_last_drum_heuristic(detections)

    left_thr, right_thr, min_cov, min_pixels = get_stricter_red_bbox_thresholds()
    stats = estimate_red_horizontal_cluster_in_bboxes(
        image_bgr,
        detections,
        min_coverage=min_cov,
        min_active_pixels=min_pixels,
    )
    if stats is None:
        return None

    x_norm = stats["x_norm"]
    base = {
        **stats,
        "left_thr": float(left_thr),
        "right_thr": float(right_thr),
        "strict_min_coverage": float(min_cov),
        "strict_min_pixels": int(min_pixels),
    }
    if x_norm >= right_thr:
        return {**base, "angle": 0, "reason": "red_bbox_right"}
    if x_norm <= left_thr:
        return {**base, "angle": 180, "reason": "red_bbox_left"}
    return None


def _register_orientation_vote(votes, vote_details, name, angle, weight, extra=None):
    if angle not in (0, 180):
        return
    weight = float(max(weight, 0.0))
    if weight <= 0.0:
        return

    a = int(angle)
    votes[a] = float(votes.get(a, 0.0) + weight)
    detail = {"angle": a, "weight": weight}
    if isinstance(extra, dict):
        detail.update(extra)
    vote_details[name] = detail


def select_dual_orientation_with_priors(dual, image_bgr=None, min_leading_zeros=LEADING_ZERO_ORIENTATION_MIN):
    z0 = leading_zero_count(dual.pred_0)
    z180 = leading_zero_count(dual.pred_180)
    conf0 = float(dual.conf_0)
    conf180 = float(dual.conf_180)

    votes = {0: 0.0, 180: 0.0}
    vote_details = {}

    red_prior = red_bbox_orientation_prior(image_bgr) if image_bgr is not None else None
    if red_prior is not None:
        _register_orientation_vote(
            votes,
            vote_details,
            "red_bbox",
            int(red_prior["angle"]),
            RED_BBOX_VOTE_WEIGHT,
            {
                "reason": red_prior.get("reason"),
                "x_norm": float(red_prior.get("x_norm", 0.0)),
                "coverage": float(red_prior.get("coverage", 0.0)),
            },
        )

    short_tail0 = bool(is_no_red_upside_down_pattern(dual.pred_0))
    short_tail180 = bool(is_no_red_upside_down_pattern(dual.pred_180))
    if red_prior is None and short_tail0 and not short_tail180:
        _register_orientation_vote(
            votes,
            vote_details,
            "no_red_short_tail_zero",
            180,
            NO_RED_SHORT_READING_VOTE_WEIGHT,
            {
                "pattern_0": True,
                "pattern_180": False,
                "max_digits": int(NO_RED_SHORT_READING_MAX_DIGITS),
            },
        )
    elif red_prior is None and short_tail180 and not short_tail0:
        _register_orientation_vote(
            votes,
            vote_details,
            "no_red_short_tail_zero",
            0,
            NO_RED_SHORT_READING_VOTE_WEIGHT,
            {
                "pattern_0": False,
                "pattern_180": True,
                "max_digits": int(NO_RED_SHORT_READING_MAX_DIGITS),
            },
        )

    tail0 = bool(is_long_tail_zero_pattern(dual.pred_0))
    tail180 = bool(is_long_tail_zero_pattern(dual.pred_180))
    if tail0 and not tail180:
        _register_orientation_vote(
            votes,
            vote_details,
            "long_tail_zero",
            180,
            STAT_TAIL_VOTE_WEIGHT,
            {"tail0": True, "tail180": False},
        )
    elif tail180 and not tail0:
        _register_orientation_vote(
            votes,
            vote_details,
            "long_tail_zero",
            0,
            STAT_TAIL_VOTE_WEIGHT,
            {"tail0": False, "tail180": True},
        )

    if z0 >= min_leading_zeros and z180 < min_leading_zeros:
        _register_orientation_vote(
            votes,
            vote_details,
            "leading_zeros",
            0,
            LEADING_ZERO_VOTE_WEIGHT,
            {"leading_zeros_0": int(z0), "leading_zeros_180": int(z180)},
        )
    elif z180 >= min_leading_zeros and z0 < min_leading_zeros:
        _register_orientation_vote(
            votes,
            vote_details,
            "leading_zeros",
            180,
            LEADING_ZERO_VOTE_WEIGHT,
            {"leading_zeros_0": int(z0), "leading_zeros_180": int(z180)},
        )

    vote_score_0 = float(votes[0])
    vote_score_180 = float(votes[180])
    if vote_score_180 > vote_score_0 + 1e-9:
        selected_angle = 180
        reason = "vote_180"
    elif vote_score_0 > vote_score_180 + 1e-9:
        selected_angle = 0
        reason = "vote_0"
    elif conf180 > conf0:
        selected_angle = 180
        reason = "confidence_180_tiebreak"
    else:
        selected_angle = 0
        reason = "confidence_0_tiebreak"

    selected_pred = dual.pred_180 if selected_angle == 180 else dual.pred_0
    selected_conf = conf180 if selected_angle == 180 else conf0

    selected_vote_contrib = {}
    for h_name, h_data in vote_details.items():
        h_angle = int(h_data.get("angle", -1))
        h_weight = float(h_data.get("weight", 0.0))
        selected_vote_contrib[h_name] = h_weight if h_angle == selected_angle else 0.0

    result = {
        "selected_pred": selected_pred,
        "selected_conf": float(selected_conf),
        "selected_angle": int(selected_angle),
        "reason": reason,
        "leading_zeros_0": int(z0),
        "leading_zeros_180": int(z180),
        "conf_0": conf0,
        "conf_180": conf180,
        "vote_score_0": vote_score_0,
        "vote_score_180": vote_score_180,
        "vote_margin": float(abs(vote_score_0 - vote_score_180)),
        "vote_total": float(vote_score_0 + vote_score_180),
        "heuristic_votes": vote_details,
        "selected_vote_contrib": selected_vote_contrib,
        "no_red_short_pattern_0": bool(short_tail0),
        "no_red_short_pattern_180": bool(short_tail180),
    }

    if red_prior is not None:
        result["red_bbox_x_norm"] = float(red_prior.get("x_norm", 0.0))
        result["red_bbox_coverage"] = float(red_prior.get("coverage", 0.0))
        result["red_bbox_left_thr"] = float(red_prior.get("left_thr", RED_CLUSTER_LEFT_MAX_X))
        result["red_bbox_right_thr"] = float(red_prior.get("right_thr", RED_CLUSTER_RIGHT_MIN_X))

    return result

In [12]:
def build_yolo11m_predictor(max_digits=MAX_READING_DIGITS):
    yolo_model = get_ultralytics_baseline_model()
    if yolo_model is None:
        raise FileNotFoundError("YOLO weights not found for ultralytics_yolo11m_baseline")

    # Runtime sanity check to fail fast if model cannot infer in current environment.
    _ = yolo_model.predict(np.zeros((64, 256, 3), dtype=np.uint8), verbose=False, imgsz=320, max_det=32)

    def pred_ultralytics_yolo_baseline(img):
        detections = extract_ultralytics_digit_detections(img, model=yolo_model)
        if not detections:
            return "", 0.0

        detections, _ = apply_ultralytics_overlap_heuristic(detections)
        detections, _ = apply_ultralytics_last_drum_heuristic(detections)
        if not detections:
            return "", 0.0

        pred_str = "".join(str(int(d["digit"])) for d in detections)
        confs = [float(d["conf"]) for d in detections]
        return pred_str, safe_mean(confs)

    predictor_pre = wrap_predictor_with_preprocess(pred_ultralytics_yolo_baseline, mode=PREPROCESS_MODE)
    return wrap_predictor_with_max_digits(predictor_pre, max_digits=max_digits)


print(f"Preprocess mode: {PREPROCESS_MODE}")
print(f"Target candidate: {TARGET_CANDIDATE}")
print(f"Target suffix: {TARGET_SUFFIX}")

Preprocess mode: none
Target candidate: ultralytics_yolo11m_baseline
Target suffix: _yolo11m_model_only


## 4. Метрики и оценка

Ниже идут вычисление Weighted Levenshtein, загрузка сплитов, Stage A и визуализация.

In [13]:
# Weighted Levenshtein error: left-side digit mistakes are penalized more than right-side ones.
# WLEV norm is computed on normalized digit strings (same normalization idea as FSA norm).
WLEV_WEIGHT_BASE = 2.0

def _positional_weights(length: int, base: float = WLEV_WEIGHT_BASE):
    if length <= 0:
        return []
    return [float(base ** (length - i - 1)) for i in range(length)]

def weighted_levenshtein_error_rate(pred: str, gt: str, base: float = WLEV_WEIGHT_BASE) -> float:
    gt_digits = digits_only(gt)
    pred_digits = digits_only(pred)

    m, n = len(gt_digits), len(pred_digits)
    if m == 0 and n == 0:
        return 0.0

    w_gt = _positional_weights(m, base=base)
    w_pred = _positional_weights(n, base=base)

    dp = np.zeros((m + 1, n + 1), dtype=np.float64)

    for i in range(1, m + 1):
        dp[i, 0] = dp[i - 1, 0] + w_gt[i - 1]
    for j in range(1, n + 1):
        dp[0, j] = dp[0, j - 1] + w_pred[j - 1]

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            del_cost = dp[i - 1, j] + w_gt[i - 1]
            ins_cost = dp[i, j - 1] + w_pred[j - 1]
            if gt_digits[i - 1] == pred_digits[j - 1]:
                sub_cost = dp[i - 1, j - 1]
            else:
                sub_cost = dp[i - 1, j - 1] + max(w_gt[i - 1], w_pred[j - 1])
            dp[i, j] = min(del_cost, ins_cost, sub_cost)

    denom = max(sum(w_gt), sum(w_pred), 1.0)
    return float(dp[m, n] / denom)

def normalize_for_wlev(text: str) -> str:
    d = digits_only(text).lstrip("0")
    return d if d else "0"

def weighted_levenshtein_norm_error_rate(pred: str, gt: str, base: float = WLEV_WEIGHT_BASE) -> float:
    pred_norm = normalize_for_wlev(pred)
    gt_norm = normalize_for_wlev(gt)
    return weighted_levenshtein_error_rate(pred_norm, gt_norm, base=base)

def mean_weighted_levenshtein_norm_error(preds, gts, base: float = WLEV_WEIGHT_BASE) -> float:
    if not preds or not gts:
        return 0.0
    return safe_mean([weighted_levenshtein_norm_error_rate(p, g, base=base) for p, g in zip(preds, gts)])

## 5. Загрузка сплитов

Подготовка train/test/all выборок для инференс-оценки.

In [14]:
# Load both train and test splits for full inference-only evaluation.
wm_poly_train = load_ocr_crops(CROPS_ROOT / "wm_polygon", "train")
wm_bbox_train = load_ocr_crops(CROPS_ROOT / "wm_bbox", "train")

wm_poly_all = wm_poly_train + wm_poly_test
wm_bbox_all = wm_bbox_train + wm_bbox_test

SPLIT_SAMPLES = {
    "train": {"wm_polygon": wm_poly_train, "wm_bbox": wm_bbox_train},
    "test": {"wm_polygon": wm_poly_test, "wm_bbox": wm_bbox_test},
    "all": {"wm_polygon": wm_poly_all, "wm_bbox": wm_bbox_all},
}

for split_name in ["train", "test", "all"]:
    split_data = SPLIT_SAMPLES[split_name]
    print(
        f"[{split_name}] wm_polygon={len(split_data['wm_polygon'])} "
        f"wm_bbox={len(split_data['wm_bbox'])}"
    )

[train] wm_polygon=870 wm_bbox=870
[test] wm_polygon=374 wm_bbox=374
[all] wm_polygon=1244 wm_bbox=1244


## 6. Stage A оценка

Запуск основного прохода по candidate и сохранение отчётов/сравнений.

In [15]:
def evaluate_predictor(predictor, samples):
    preds, gts, times = [], [], []
    non_empty, chosen_180 = 0, 0
    leading_zero_overrides = 0
    red_bbox_overrides = 0
    long_tail_zero_overrides = 0
    confidence_tiebreaks = 0

    for img_path, gt in samples:
        image = cv2.imread(str(img_path))
        if image is None:
            continue

        t0 = time.perf_counter()
        dual = dual_read_inference(image, predictor)
        selected = select_dual_orientation_with_priors(dual, image_bgr=image)

        times.append((time.perf_counter() - t0) * 1000.0)
        preds.append(selected["selected_pred"])
        gts.append(gt)

        non_empty += int(selected["selected_pred"] != "")
        chosen_180 += int(selected["selected_angle"] == 180)

        contrib = selected.get("selected_vote_contrib", {})
        leading_zero_overrides += int(float(contrib.get("leading_zeros", 0.0)) > 0.0)
        red_bbox_overrides += int(float(contrib.get("red_bbox", 0.0)) > 0.0)
        long_tail_zero_overrides += int(float(contrib.get("long_tail_zero", 0.0)) > 0.0)
        confidence_tiebreaks += int(str(selected.get("reason", "")).startswith("confidence_"))

    m = evaluate_ocr_batch(preds, gts) if gts else {"fsa_raw": 0.0, "fsa_norm": 0.0, "pda": 0.0, "cer": 1.0}
    wlev_norm_mean = mean_weighted_levenshtein_norm_error(preds, gts, base=WLEV_WEIGHT_BASE)
    return {
        **m,
        "wlev_norm_mean": float(wlev_norm_mean),
        "avg_ms": safe_mean(times),
        "non_empty_rate": float(non_empty / max(len(gts), 1)),
        "selected_180_share": float(chosen_180 / max(len(gts), 1)),
        "leading_zero_override_rate": float(leading_zero_overrides / max(len(gts), 1)),
        "red_bbox_override_rate": float(red_bbox_overrides / max(len(gts), 1)),
        "long_tail_zero_override_rate": float(long_tail_zero_overrides / max(len(gts), 1)),
        "confidence_tiebreak_rate": float(confidence_tiebreaks / max(len(gts), 1)),
    }


if "SPLIT_SAMPLES" not in globals():
    wm_poly_train = load_ocr_crops(CROPS_ROOT / "wm_polygon", "train")
    wm_bbox_train = load_ocr_crops(CROPS_ROOT / "wm_bbox", "train")
    SPLIT_SAMPLES = {
        "train": {"wm_polygon": wm_poly_train, "wm_bbox": wm_bbox_train},
        "test": {"wm_polygon": wm_poly_test, "wm_bbox": wm_bbox_test},
        "all": {"wm_polygon": wm_poly_train + wm_poly_test, "wm_bbox": wm_bbox_train + wm_bbox_test},
    }

MIN_NON_EMPTY_RATE = 0.10
MAX_AVG_MS = 1500.0

strict_left_thr, strict_right_thr, strict_min_cov, strict_min_pixels = get_stricter_red_bbox_thresholds()

predictor = build_yolo11m_predictor(max_digits=MAX_READING_DIGITS)
effective_preprocess_mode = "none"

rows = []
split_metrics = {}
for split_name in ["train", "test", "all"]:
    split_data = SPLIT_SAMPLES[split_name]
    poly = evaluate_predictor(predictor, split_data["wm_polygon"])
    bbox = evaluate_predictor(predictor, split_data["wm_bbox"])
    pass_stage_a = (
        poly["non_empty_rate"] >= MIN_NON_EMPTY_RATE
        and bbox["non_empty_rate"] >= MIN_NON_EMPTY_RATE
        and poly["avg_ms"] <= MAX_AVG_MS
        and bbox["avg_ms"] <= MAX_AVG_MS
    )

    split_metrics[split_name] = {
        "wm_polygon": poly,
        "wm_bbox": bbox,
        "stage_a_pass": bool(pass_stage_a),
    }

    rows.append(
        {
            "candidate": TARGET_CANDIDATE,
            "split": split_name,
            "preprocess_mode": effective_preprocess_mode,
            "stage_a_pass": bool(pass_stage_a),
            "poly_fsa_norm": poly["fsa_norm"],
            "bbox_fsa_norm": bbox["fsa_norm"],
            "poly_wlev_norm_mean": poly["wlev_norm_mean"],
            "bbox_wlev_norm_mean": bbox["wlev_norm_mean"],
            "poly_ms": poly["avg_ms"],
            "bbox_ms": bbox["avg_ms"],
        }
    )

report = {
    "run_date": datetime.now().isoformat(),
    "candidate": TARGET_CANDIDATE,
    "preprocess_mode": effective_preprocess_mode,
    "max_reading_digits": MAX_READING_DIGITS,
    "leading_zero_orientation_min": LEADING_ZERO_ORIENTATION_MIN,
    "long_tail_zero_min_digits": LONG_TAIL_ZERO_MIN_DIGITS,
    "long_tail_zero_min_suffix": LONG_TAIL_ZERO_MIN_SUFFIX,
    "red_cluster_min_coverage": RED_CLUSTER_MIN_COVERAGE,
    "red_cluster_left_max_x": RED_CLUSTER_LEFT_MAX_X,
    "red_cluster_right_min_x": RED_CLUSTER_RIGHT_MIN_X,
    "red_bbox_min_active_pixels": RED_BBOX_MIN_ACTIVE_PIXELS,
    "red_bbox_vote_weight": RED_BBOX_VOTE_WEIGHT,
    "stat_tail_vote_weight": STAT_TAIL_VOTE_WEIGHT,
    "leading_zero_vote_weight": LEADING_ZERO_VOTE_WEIGHT,
    "red_bbox_strictness": RED_BBOX_STRICTNESS,
    "red_bbox_strict_left_max_x": float(strict_left_thr),
    "red_bbox_strict_right_min_x": float(strict_right_thr),
    "red_bbox_strict_min_coverage": float(strict_min_cov),
    "red_bbox_strict_min_pixels": int(strict_min_pixels),
    "wlev_weight_base": float(WLEV_WEIGHT_BASE),
    "splits": split_metrics,
    "wm_polygon": split_metrics["test"]["wm_polygon"],
    "wm_bbox": split_metrics["test"]["wm_bbox"],
    "stage_a_pass": bool(split_metrics["test"]["stage_a_pass"]),
}

comparison_csv_path = RESULTS_DIR / f"ocr_comparison{TARGET_SUFFIX}.csv"
if split_metrics["test"]["stage_a_pass"]:
    poly_test = split_metrics["test"]["wm_polygon"]
    bbox_test = split_metrics["test"]["wm_bbox"]
    row = build_ocr_comparison_row(
        method=f"pretrained_{TARGET_CANDIDATE}{TARGET_SUFFIX}",
        wm_poly_metrics={"fsa_raw": poly_test["fsa_raw"], "fsa_norm": poly_test["fsa_norm"], "pda": poly_test["pda"], "cer": poly_test["cer"]},
        wm_bbox_metrics={"fsa_raw": bbox_test["fsa_raw"], "fsa_norm": bbox_test["fsa_norm"], "pda": bbox_test["pda"], "cer": bbox_test["cer"]},
        wm_poly_ms=poly_test["avg_ms"],
        wm_bbox_ms=bbox_test["avg_ms"],
        run_date=datetime.now().strftime("%Y-%m-%d %H:%M"),
    )
    append_ocr_comparison_row(comparison_csv_path, row)

report_path = RESULTS_DIR / f"ocr_pretrained_pilot{TARGET_SUFFIX}.json"
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

display(pd.DataFrame(rows))
print("Preprocess mode:", effective_preprocess_mode)
print("Weighted Levenshtein norm base:", WLEV_WEIGHT_BASE)
print("Candidate:", TARGET_CANDIDATE)
print("Saved:", report_path)
if split_metrics["test"]["stage_a_pass"]:
    print("Updated comparison table:", comparison_csv_path)


,candidate,split,preprocess_mode,stage_a_pass,poly_fsa_norm,bbox_fsa_norm,poly_wlev_norm_mean,bbox_wlev_norm_mean,poly_ms,bbox_ms
0,ultralytics_yolo11m_baseline,train,none,True,0.751724,0.854023,0.051729,0.034382,37.283874,37.798701
1,ultralytics_yolo11m_baseline,test,none,True,0.796791,0.882353,0.040593,0.018580,40.346828,37.574951
2,ultralytics_yolo11m_baseline,all,none,True,0.765273,0.862540,0.048381,0.029631,41.442801,43.425195


Preprocess mode: none
Weighted Levenshtein norm base: 2.0
Candidate: ultralytics_yolo11m_baseline
Saved: C:\Users\alike\WaterMeterCV\results\ocr_pretrained_pilot_yolo11m_model_only.json
Updated comparison table: C:\Users\alike\WaterMeterCV\results\ocr_comparison_yolo11m_model_only.csv


## 7. Визуальная проверка

Построение превью с подсветкой ошибок для быстрого sanity-check результатов.

In [16]:
import matplotlib.pyplot as plt

VIS_N = 100
VIS_DPI_DEFAULT = 140
VIS_DPI_ULTRALYTICS = 220


def _normalize_label(v: str) -> str:
    s = str(v)
    t = s.lstrip("0")
    return t if t else "0"


def _predict_ultralytics_digit_boxes(candidate_name: str, image_bgr):
    if candidate_name != TARGET_CANDIDATE:
        return []

    detections = extract_ultralytics_digit_detections(image_bgr)
    if not detections:
        return []

    detections, _ = apply_ultralytics_overlap_heuristic(detections)
    detections, _ = apply_ultralytics_last_drum_heuristic(detections)
    return detections


def _draw_error_bboxes(image_bgr, detections):
    vis = image_bgr.copy()
    h, w = vis.shape[:2]

    for det in detections:
        x1 = int(round(det["x1"]))
        y1 = int(round(det["y1"]))
        x2 = int(round(det["x2"]))
        y2 = int(round(det["y2"]))
        x1 = max(0, min(w - 1, x1))
        y1 = max(0, min(h - 1, y1))
        x2 = max(0, min(w - 1, x2))
        y2 = max(0, min(h - 1, y2))

        if x2 <= x1 or y2 <= y1:
            continue

        cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 0, 255), 1)

    return vis


def visualize_candidate(
    candidate_name: str,
    predictor,
    n: int = VIS_N,
    preprocess_mode: str = PREPROCESS_MODE,
    apply_preprocess: bool = False,
):
    fig, axes = plt.subplots(2, n, figsize=(2.2 * n, 6.6), squeeze=False)

    rows = [
        ("wm_polygon", wm_poly_test),
        ("wm_bbox", wm_bbox_test),
    ]

    for row_i, (crop_key, samples) in enumerate(rows):
        for col_i in range(n):
            ax = axes[row_i, col_i]
            if col_i >= len(samples):
                ax.axis("off")
                continue

            img_path, gt = samples[col_i]
            image = cv2.imread(str(img_path))
            if image is None:
                ax.axis("off")
                continue

            dual = dual_read_inference(image, predictor)
            selected = select_dual_orientation_with_priors(dual, image_bgr=image)
            pred = selected.get("selected_pred", "")
            selected_angle = int(selected.get("selected_angle", 0))

            display_image = cv2.rotate(image, cv2.ROTATE_180) if selected_angle == 180 else image
            preview_img = preprocess_for_ocr(display_image, mode=preprocess_mode) if apply_preprocess else display_image.copy()

            gt_norm = _normalize_label(gt)
            pred_norm = _normalize_label(pred)
            ok_norm = pred_norm == gt_norm
            title_color = "green" if ok_norm else "red"

            if not ok_norm:
                error_boxes = _predict_ultralytics_digit_boxes(candidate_name, display_image)
                if error_boxes:
                    preview_img = _draw_error_bboxes(preview_img, error_boxes)

            sample_id = img_path.stem
            shown_pred = pred if pred else "EMPTY"
            ax.imshow(cv2.cvtColor(preview_img, cv2.COLOR_BGR2RGB))
            ax.set_title(f"ID={sample_id}\nGT={gt} | P={shown_pred}", fontsize=8, pad=2, color=title_color)
            if col_i == 0:
                ax.set_ylabel(crop_key, fontsize=9, fontweight="bold")
            ax.axis("off")

    effective_mode = preprocess_mode if apply_preprocess else "none"
    fig.suptitle(
        f"Pretrained OCR preview: {candidate_name} | preprocess={effective_mode}",
        fontsize=13,
    )
    fig.tight_layout(rect=[0.0, 0.03, 1.0, 0.93])

    out_path = RESULTS_DIR / f"ocr_pretrained_preview_{candidate_name}_{effective_mode}{TARGET_SUFFIX}.png"
    save_dpi = VIS_DPI_ULTRALYTICS if candidate_name == TARGET_CANDIDATE else VIS_DPI_DEFAULT
    fig.savefig(out_path, dpi=save_dpi)
    plt.close(fig)
    print(f"Saved preview: {out_path} (dpi={save_dpi})")


predictor_for_vis = build_yolo11m_predictor(max_digits=MAX_READING_DIGITS)
print(f"Visualizing candidate: {TARGET_CANDIDATE} | preprocess=none")
visualize_candidate(
    TARGET_CANDIDATE,
    predictor_for_vis,
    n=VIS_N,
    preprocess_mode=PREPROCESS_MODE,
    apply_preprocess=False,
)


Visualizing candidate: ultralytics_yolo11m_baseline | preprocess=none
Saved preview: C:\Users\alike\WaterMeterCV\results\ocr_pretrained_preview_ultralytics_yolo11m_baseline_none_yolo11m_model_only.png (dpi=220)


## (YOLO11m)

A candidate passes Stage A if both crop paths provide non-empty predictions and acceptable latency.
This notebook is the base for next pipeline steps; extended sweeps and artifact-vs-artifact comparisons are moved to separate notebooks.
